# 1.1. Install and import dependencies

In [1]:
!pip install poetry --quiet

In [2]:
import os

# Write pyproject.toml at the repo root
repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
os.chdir(repo_root)

pyproject_content = """\
[tool.poetry]
name = "ml-engineering-lab01"
description = "CIFAR-10 image classification training pipeline"
authors = ["Yehor <yehor.chukanov@cs.khpi.edu.ua>"]
version = "0.1.0"

[tool.poetry.dependencies]
python = "~3.11"
torch = {version = ">=2.5.0", source = "pytorch"}
torchvision = {version = ">=0.20.0", source = "pytorch"}
tqdm = ">=4.66"
matplotlib = ">=3.8"
numpy = ">=1.24,<2.0"
pyyaml = ">=6.0"
pandas = ">=2.0"
requests = ">=2.31"
scikit-learn = ">=1.3"

[tool.poetry.group.dev.dependencies]
mypy = ">=1.8"
ruff = ">=0.2"
black = ">=24.0"
isort = ">=5.13"
ipykernel = ">=6.29"

[[tool.poetry.source]]
name = "pytorch"
priority = "supplemental"
url = "https://download.pytorch.org/whl/cu128"

[tool.black]
line-length = 100

[tool.isort]
profile = "black"
line_length = 100

[tool.ruff]
line-length = 100

[tool.mypy]
python_version = "3.11"
warn_return_any = true
warn_unused_configs = true

[build-system]
requires = ["poetry-core>=1.0.0"]
build-backend = "poetry.core.masonry.api"
"""

with open("pyproject.toml", "w") as f:
    f.write(pyproject_content)

print(f"Created pyproject.toml at {repo_root}")

# Install dependencies
!poetry install --no-root 2>&1 | tail -5

Created pyproject.toml at /home/yehor/study/ml_engineering
Installing dependencies from lock file

No dependencies to install or update


Load configuration from YAML and set up imports.

In [3]:
import logging
import pickle
import tarfile
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.optim as optim
import yaml
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"Using device: {device}")

In [4]:
# Fix logging in notebooks
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

logging.basicConfig(level=logging.INFO)

# Load config
config_path = Path("labs/lab01/configs/config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

logging.info(f"Config loaded from {config_path}")
config

INFO:root:Config loaded from labs/lab01/configs/config.yaml


{'data': {'dataset_url': 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz',
  'data_dir': 'data',
  'val_size': 0.2,
  'random_state': 42},
 'model': {'n_classes': 10},
 'training': {'batch_size': 128,
  'num_workers': 4,
  'num_epochs': 20,
  'learning_rate': 0.001},
 'artifacts': {'save_dir': 'artifacts',
  'best_model_filename': 'best_model.pth'}}

# 1.2 Data download

In [5]:
def download_and_extract(url: str, save_dir: str, filename: Optional[str] = None) -> str:
    """
    Downloads a file from the given URL and extracts it if it is an archive.

    Args:
        url: The URL of the file to download.
        save_dir: Directory where the file will be saved and extracted.
        filename: Optional filename. If None, uses the filename from the URL.

    Returns:
        The full path to the directory containing the extracted files.
    """
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    if filename is None:
        filename = url.split("/")[-1]

    file_path = save_path / filename
    result_path = str(save_path)

    # Download if file does not exist
    if not file_path.exists():
        logging.info(f"Downloading '{filename}' from '{url}'...")
        try:
            response = requests.get(url, timeout=120)
            response.raise_for_status()
            with open(file_path, "wb") as f:
                f.write(response.content)
            logging.info(f"Download successful. File saved to: '{file_path}'")
        except requests.RequestException as e:
            logging.error(f"Error downloading file from {url}: {e}")
            raise

    # Extract if archive exists
    try:
        if file_path.exists() and filename.endswith((".tar.gz", ".tgz")):
            logging.info(f"Extracting '{filename}'...")
            with tarfile.open(file_path, "r:gz") as tar_ref:
                tar_ref.extractall(save_path)
            file_path.unlink()
        elif file_path.exists() and filename.endswith(".tar"):
            logging.info(f"Extracting '{filename}'...")
            with tarfile.open(file_path, "r") as tar_ref:
                tar_ref.extractall(save_path)
            file_path.unlink()
        else:
            logging.info(f"File '{filename}' already processed or not an archive.")
    except tarfile.ReadError as e:
        logging.error(f"Error extracting file '{filename}': {e}")
        raise

    return result_path

# 1.3 Data ingestion

In [6]:
def _unpickle(file_path: str) -> Dict[bytes, Any]:
    """Load a CIFAR-10 batch file."""
    with open(file_path, "rb") as f:
        batch = pickle.load(f, encoding="bytes")
    return batch  # type: ignore[no-any-return]


def load_cifar10(data_dir: str) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Load all CIFAR-10 train and test batches from the extracted directory.

    Returns:
        Tuple of (train_images, train_labels, test_images, test_labels).
        Images shape: (N, 3, 32, 32), dtype float32, normalized to [0, 1].
    """
    cifar_dir = Path(data_dir) / "cifar-10-batches-py"

    # Load training batches
    train_images_list = []
    train_labels_list = []
    for i in range(1, 6):
        batch = _unpickle(str(cifar_dir / f"data_batch_{i}"))
        train_images_list.append(batch[b"data"])
        train_labels_list.extend(batch[b"labels"])

    train_images = np.concatenate(train_images_list).reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
    train_labels = np.array(train_labels_list, dtype=np.int64)

    # Load test batch
    test_batch = _unpickle(str(cifar_dir / "test_batch"))
    test_images = test_batch[b"data"].reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
    test_labels = np.array(test_batch[b"labels"], dtype=np.int64)

    logging.info(
        f"Loaded CIFAR-10: train={train_images.shape[0]}, test={test_images.shape[0]}"
    )
    return train_images, train_labels, test_images, test_labels


def train_val_split(
    images: np.ndarray,
    labels: np.ndarray,
    val_size: float = 0.2,
    random_state: int = 42,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Split arrays into train and validation subsets."""
    rng = np.random.RandomState(random_state)
    n = len(images)
    indices = rng.permutation(n)
    val_count = int(n * val_size)

    val_idx = indices[:val_count]
    train_idx = indices[val_count:]

    return images[train_idx], labels[train_idx], images[val_idx], labels[val_idx]


def load_labels(data_dir: str) -> pd.DataFrame:
    """
    Load CIFAR-10 labels into a DataFrame with image index and label columns.

    Args:
        data_dir: Path to extracted CIFAR-10 data.

    Returns:
        DataFrame with columns: image_path, label.
    """
    train_images, train_labels, test_images, test_labels = load_cifar10(data_dir)

    all_labels = np.concatenate([train_labels, test_labels])
    all_paths = [f"train_{i}" for i in range(len(train_labels))] + \
                [f"test_{i}" for i in range(len(test_labels))]

    labels_df = pd.DataFrame({"image_path": all_paths, "label": all_labels})
    return labels_df


def process_data(
    data_dir: str, config: Dict[str, Any]
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Load CIFAR-10 and split into train/val/test numpy arrays.

    Returns:
        (train_images, train_labels, val_images, val_labels, test_images, test_labels)
    """
    train_images, train_labels, test_images, test_labels = load_cifar10(data_dir)

    train_images, train_labels, val_images, val_labels = train_val_split(
        train_images,
        train_labels,
        val_size=config["data"]["val_size"],
        random_state=config["data"]["random_state"],
    )

    logging.info(
        f"Data splits: train={len(train_labels)}, val={len(val_labels)}, test={len(test_labels)}"
    )
    return train_images, train_labels, val_images, val_labels, test_images, test_labels

# 1.4 Training loop

## 1.4.1 Data loaders

In [7]:
class CifarDataset(Dataset):
    """Dataset for CIFAR-10 numpy arrays."""

    def __init__(
        self,
        images: np.ndarray,
        labels: np.ndarray,
        transform: Optional[nn.Module] = None,
    ) -> None:
        self.images = torch.from_numpy(images)
        self.labels = torch.from_numpy(labels)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        image = self.images[idx]
        label = int(self.labels[idx])

        if self.transform:
            image = self.transform(image)

        return image, label


def create_data_loader(
    images: np.ndarray,
    labels: np.ndarray,
    config: Dict[str, Any],
    transform: Optional[nn.Module] = None,
    shuffle: bool = True,
) -> DataLoader:
    """Create a DataLoader from numpy arrays."""
    dataset = CifarDataset(images, labels, transform=transform)
    data_loader = DataLoader(
        dataset,
        batch_size=config["training"]["batch_size"],
        shuffle=shuffle,
        num_workers=config["training"]["num_workers"],
    )
    return data_loader

## 1.4.2. Model, loss and optimizers

In [8]:
class CifarCNN(nn.Module):
    """Small CNN for CIFAR-10 classification."""

    def __init__(self, n_classes: int) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = self.classifier(x)
        return x

## 1.4.3. Training loop function

In [9]:
def train_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    loss_function: nn.Module,
    optimizer: optim.Optimizer,
    num_epochs: int,
    device: torch.device,
    save_path: Path = Path("best_model.pth"),
) -> Path:
    """Train the model and save the best checkpoint based on validation loss."""
    model.to(device)
    best_val_loss: float = float("inf")
    best_model_path: Path = save_path

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        for batch_inputs, batch_targets in train_loader:
            batch_inputs = batch_inputs.to(device)
            batch_targets = batch_targets.to(device)

            outputs = model(batch_inputs)
            loss = loss_function(outputs, batch_targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        logging.info(f"Epoch {epoch + 1}/{num_epochs}, Train Loss: {avg_train_loss:.4f}")

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for val_inputs, val_targets in val_loader:
                val_inputs = val_inputs.to(device)
                val_targets = val_targets.to(device)
                val_outputs = model(val_inputs)
                val_loss += loss_function(val_outputs, val_targets).item()

        val_loss /= len(val_loader)
        logging.info(f"Epoch {epoch + 1}/{num_epochs}, Val Loss: {val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_model_path)
            logging.info(f"Best model saved (val_loss={best_val_loss:.4f})")

    logging.info("Training complete.")
    return best_model_path

# 1.5 Test loop

In [10]:
def test_model(
    model: nn.Module,
    test_loader: DataLoader,
    loss_function: nn.Module,
    device: torch.device,
) -> Dict[str, float]:
    """
    Evaluate model on test set. Returns loss, accuracy, precision, recall, F1.
    """
    model.eval()
    test_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)
            test_loss += loss_function(outputs, targets).item()

            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(targets.cpu().numpy())

    test_loss /= len(test_loader)
    all_preds_arr = np.array(all_preds)
    all_targets_arr = np.array(all_targets)

    metrics = {
        "test_loss": test_loss,
        "accuracy": accuracy_score(all_targets_arr, all_preds_arr),
        "precision": precision_score(all_targets_arr, all_preds_arr, average="weighted"),
        "recall": recall_score(all_targets_arr, all_preds_arr, average="weighted"),
        "f1": f1_score(all_targets_arr, all_preds_arr, average="weighted"),
    }

    for name, value in metrics.items():
        logging.info(f"{name}: {value:.4f}")

    return metrics

# 1.6. Put everything together

In [11]:
def main() -> Dict[str, float]:
    # Step 1: Download data
    data_dir = download_and_extract(
        url=config["data"]["dataset_url"],
        save_dir=config["data"]["data_dir"],
    )

    # Step 2: Load and split data
    train_images, train_labels, val_images, val_labels, test_images, test_labels = process_data(
        data_dir, config
    )

    # Step 3: Data augmentation for training
    train_transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(32, padding=4),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]),
    ])
    eval_transform = transforms.Compose([
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]),
    ])

    train_loader = create_data_loader(train_images, train_labels, config, transform=train_transform)
    val_loader = create_data_loader(val_images, val_labels, config, transform=eval_transform, shuffle=False)
    test_loader = create_data_loader(test_images, test_labels, config, transform=eval_transform, shuffle=False)

    # Step 4: Define model, loss, optimizer
    model = CifarCNN(n_classes=config["model"]["n_classes"]).to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config["training"]["learning_rate"])

    # Step 5: Train
    artifacts_dir = Path(config["artifacts"]["save_dir"])
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    save_path = artifacts_dir / config["artifacts"]["best_model_filename"]

    best_model_path = train_model(
        model, train_loader, val_loader, loss_function, optimizer,
        num_epochs=config["training"]["num_epochs"],
        device=device,
        save_path=save_path,
    )

    # Step 6: Load best model and test
    model.load_state_dict(torch.load(best_model_path, weights_only=True))
    metrics = test_model(model, test_loader, loss_function, device)

    return metrics

In [12]:
results = main()

INFO:root:Extracting 'cifar-10-python.tar.gz'...
INFO:root:Loaded CIFAR-10: train=50000, test=10000
INFO:root:Data splits: train=40000, val=10000, test=10000
INFO:root:Epoch 1/20, Train Loss: 1.9057
INFO:root:Epoch 1/20, Val Loss: 1.4861
INFO:root:Best model saved (val_loss=1.4861)
INFO:root:Epoch 2/20, Train Loss: 1.6956
INFO:root:Epoch 2/20, Val Loss: 1.3742
INFO:root:Best model saved (val_loss=1.3742)
INFO:root:Epoch 3/20, Train Loss: 1.6006
INFO:root:Epoch 3/20, Val Loss: 1.3218
INFO:root:Best model saved (val_loss=1.3218)
INFO:root:Epoch 4/20, Train Loss: 1.5296
INFO:root:Epoch 4/20, Val Loss: 1.1682
INFO:root:Best model saved (val_loss=1.1682)
INFO:root:Epoch 5/20, Train Loss: 1.4882
INFO:root:Epoch 5/20, Val Loss: 1.1387
INFO:root:Best model saved (val_loss=1.1387)
INFO:root:Epoch 6/20, Train Loss: 1.4433
INFO:root:Epoch 6/20, Val Loss: 1.0893
INFO:root:Best model saved (val_loss=1.0893)
INFO:root:Epoch 7/20, Train Loss: 1.4001
INFO:root:Epoch 7/20, Val Loss: 1.0685
INFO:root:Be